# Uniwersalna analiza modelu Ly-alpha P1D

Ten notebook nie zawiera na sztywno benchmarku żadnej publikacji. Zmień ścieżki w komórce konfiguracji, aby przeanalizować dowolny `TheoryBundle`. Domyślny przykład używa zweryfikowanego LCDM 2022.

## 1. Importy i katalog projektu

Z katalogu projektu wykonaj wcześniej `python -m pip install -e '.[notebook]'`. CLASS nie jest potrzebny.

In [ ]:
from __future__ import annotations

from pathlib import Path
import json
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

plt.style.use('seaborn-v0_8-whitegrid')
pd.set_option('display.max_columns', None)
candidates = [Path.cwd(), Path.cwd().parent, Path.cwd() / 'lyalpha_one_loop', Path.cwd().parent / 'lyalpha_one_loop']
PROJECT_ROOT = next((p.resolve() for p in candidates if (p / 'pyproject.toml').is_file()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError('Nie znaleziono katalogu projektu z pyproject.toml.')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from lyalpha_pt.data import load_dr12
from lyalpha_pt.fit import EffectiveModelFit, PARAMETER_NAMES, write_fit_result
from lyalpha_pt.theory import CHANNELS, load_theory
print('PROJECT_ROOT =', PROJECT_ROOT)

## 2. Konfiguracja użytkownika

Dla nowego modelu zwykle wystarczy zmienić `THEORY_PATH`, `FIT_PATH`, `MODEL_LABEL` i ewentualnie ustawić `RUN_NEW_FITS=True`.

In [ ]:
THEORY_PATH = PROJECT_ROOT / 'results' / 'lcdm_2022_planck_precision_v3_3.npz'
FIT_PATH = PROJECT_ROOT / 'results' / 'lcdm_2022_planck_precision_v3_3_fit.json'
MODEL_LABEL = None  # None -> nazwa zapisana w bundle'u
RUN_NEW_FITS = False
K_UV_CUT = 20.0
COVARIANCE_MODE = 'paper_diag'
FIT_SEEDS = (12345, 23456, 34567, 45678, 56789)
EXPAND_COUNTERTERM = True
OUTPUT_DIR = None  # None -> results/<model>_analysis

if not THEORY_PATH.is_file():
    raise FileNotFoundError(THEORY_PATH)
if not RUN_NEW_FITS and not FIT_PATH.is_file():
    raise FileNotFoundError('Brak FIT_PATH; ustaw RUN_NEW_FITS=True albo wskaż istniejący JSON.')

## 3. Bundle i dane

In [ ]:
theory = load_theory(THEORY_PATH)
dataset = load_dr12(PROJECT_ROOT / 'data')
MODEL_LABEL = MODEL_LABEL or theory.metadata['model']['name']
OUTPUT_DIR = Path(OUTPUT_DIR) if OUTPUT_DIR is not None else PROJECT_ROOT / 'results' / f'{MODEL_LABEL}_analysis'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
meta = theory.metadata
numerics = meta['numerics']
derived = meta.get('derived_cosmology', {})

bundle_table = pd.DataFrame([
    {'quantity': 'model', 'value': meta['model']['name']},
    {'quantity': 'description', 'value': meta['model'].get('description', '')},
    {'quantity': 'bundle digest', 'value': meta['bundle_digest']},
    {'quantity': 'loop formula', 'value': meta['loop_formula']},
    {'quantity': 'loop source', 'value': meta['loop_source']},
    {'quantity': 'loop weight', 'value': theory.loop_weight},
    {'quantity': 'P13 method', 'value': meta['p13_method']},
    {'quantity': 'k loop [h/Mpc]', 'value': f'{theory.k_loop[0]:.4g}--{theory.k_loop[-1]:.4g}'},
    {'quantity': 'k_trust [h/Mpc]', 'value': numerics['k_trust']},
    {'quantity': 'q_max [h/Mpc]', 'value': numerics['q_max']},
    {'quantity': 'redshifts', 'value': ', '.join(f'{z:.1f}' for z in theory.z)},
    {'quantity': 'N data', 'value': dataset.n_data},
] + [{'quantity': name, 'value': value} for name, value in derived.items()])
display(bundle_table)
bundle_table.to_csv(OUTPUT_DIR / 'bundle_summary.csv', index=False)

## 4. Wczytanie albo ponowne wykonanie fitu

Nowy fit używa tej samej procedury co `scripts/fit_p1d.py`: bazowy box, opcjonalna kontynuacja do szerokiego countertermu oraz ścieżka cutoffów 10 → 15 → 20.

In [ ]:
def fit_mode_with_path(mode, initial_thetas=()):
    path = [K_UV_CUT] if K_UV_CUT <= 10 else [10.0] + ([15.0] if K_UV_CUT > 15 else []) + [K_UV_CUT]
    path = list(dict.fromkeys(path))
    result = None
    history = []
    for index, cutoff in enumerate(path):
        fitter = EffectiveModelFit(dataset, theory, mode=mode, k_uv_cut=cutoff, covariance_mode=COVARIANCE_MODE)
        if index == 0:
            result = fitter.fit_staged(seeds=FIT_SEEDS, expand_counterterm=EXPAND_COUNTERTERM, initial_thetas=initial_thetas)
        else:
            result = fitter.continue_staged(result.x, expand_counterterm=EXPAND_COUNTERTERM)
        history.append({'k_uv_cut_hmpc': cutoff, 'chi2': fitter.chi2(result.x)})
    return fitter, result, history

if RUN_NEW_FITS:
    linear_fitter, linear_result, linear_path = fit_mode_with_path('linear')
    loop_fitter, loop_result, loop_path = fit_mode_with_path('one_loop', [linear_result.x])
    fitters = {'linear': linear_fitter, 'one_loop': loop_fitter}
    results = {'linear': linear_result, 'one_loop': loop_result}
    records = {mode: fitters[mode].result_record(results[mode]) for mode in fitters}
    records['linear']['cutoff_continuation'] = linear_path
    records['one_loop']['cutoff_continuation'] = loop_path
    saved = {
        'theory_file': str(THEORY_PATH.resolve()),
        'data_dir': str((PROJECT_ROOT / 'data').resolve()),
        'optimizer_parameterization': 'four_dimensional_with_two_profiled_amplitudes',
        'cutoff_continuation_enabled': True,
        'fits': records,
        'delta_chi2_one_loop_minus_linear': records['one_loop']['chi2'] - records['linear']['chi2'],
    }
    write_fit_result(FIT_PATH, saved)
else:
    saved = json.loads(FIT_PATH.read_text())
    fitters = {mode: EffectiveModelFit(dataset, theory, mode=mode, k_uv_cut=K_UV_CUT, covariance_mode=COVARIANCE_MODE) for mode in ('linear', 'one_loop')}
    for mode in ('linear', 'one_loop'):
        record = saved['fits'][mode]
        if record.get('theory_digest') != meta['bundle_digest']:
            raise ValueError(f'{mode}: fit JSON pochodzi z innego bundle-a.')
        if record.get('covariance_mode') != COVARIANCE_MODE:
            raise ValueError(f'{mode}: inny wariant kowariancji w fit JSON.')

theta = {mode: np.array([saved['fits'][mode]['parameters'][name] for name in PARAMETER_NAMES]) for mode in ('linear', 'one_loop')}
summaries = {mode: fitters[mode].summary(theta[mode]) for mode in theta}
chi2_linear = summaries['linear']['chi2']
chi2_loop = summaries['one_loop']['chi2']
print(f'linear   : {chi2_linear:.9f}')
print(f'one-loop : {chi2_loop:.9f}')
print(f'Delta    : {chi2_loop - chi2_linear:.9f}')

## 5. Parametry, granice i wkłady redshiftów

In [ ]:
parameter_table = pd.DataFrame({
    'parameter': [*PARAMETER_NAMES, 'alpha_F'],
    'linear': [*theta['linear'], np.exp(theta['linear'][0])],
    'one_loop': [*theta['one_loop'], np.exp(theta['one_loop'][0])],
})
display(parameter_table)
parameter_table.to_csv(OUTPUT_DIR / 'best_fit_parameters.csv', index=False)

boundary_rows = []
for mode in ('linear', 'one_loop'):
    for row in saved['fits'][mode].get('boundary_diagnostics', []):
        boundary_rows.append({'mode': mode, **row})
boundary_table = pd.DataFrame(boundary_rows)
display(boundary_table)
boundary_table.to_csv(OUTPUT_DIR / 'boundary_diagnostics.csv', index=False)

per_z = []
for linear_row, loop_row in zip(fitters['linear'].chi2_by_redshift(theta['linear']), fitters['one_loop'].chi2_by_redshift(theta['one_loop'])):
    per_z.append({'z': linear_row['z'], 'N': linear_row['n_data'], 'chi2_linear': linear_row['chi2'], 'chi2_one_loop': loop_row['chi2'], 'Delta_chi2': loop_row['chi2'] - linear_row['chi2']})
per_z_table = pd.DataFrame(per_z)
display(per_z_table)
per_z_table.to_csv(OUTPUT_DIR / 'chi2_by_redshift.csv', index=False)

## 6. Dane i najlepsze krzywe

In [ ]:
predictions = {mode: fitters[mode].model(theta[mode]) for mode in theta}
sigma = np.sqrt(np.diag(fitters['linear'].covariance))
fig, axes = plt.subplots(4, 2, figsize=(15, 17), sharex=True)
axes = axes.ravel()
for axis, row in zip(axes, per_z):
    z = row['z']; mask = np.isclose(dataset.z, z); order = np.argsort(dataset.k_velocity[mask])
    kv = dataset.k_velocity[mask][order]
    axis.errorbar(kv, (kv * dataset.p1d[mask] / np.pi)[order], yerr=(kv * sigma[mask] / np.pi)[order], fmt='o', ms=4, color='black', ecolor='0.6', label='BOSS')
    axis.plot(kv, (kv * predictions['linear'][mask] / np.pi)[order], '--', lw=2, label='linear')
    axis.plot(kv, (kv * predictions['one_loop'][mask] / np.pi)[order], '-', lw=2, label='one-loop')
    axis.set_xscale('log'); axis.set_title(f"z={z:.1f}: chi2 lin={row['chi2_linear']:.2f}, 1L={row['chi2_one_loop']:.2f}")
    axis.set_ylabel(r'$k_v P_{1D}/\pi$')
    axis.legend(fontsize=8)
for axis in axes[len(per_z):]: axis.set_visible(False)
for axis in axes[-2:]: axis.set_xlabel(r'$k_v$ [s/km]')
fig.suptitle(f'{MODEL_LABEL}: chi2 lin={chi2_linear:.3f}, 1L={chi2_loop:.3f}, Delta={chi2_loop-chi2_linear:.3f}', y=0.995)
fig.tight_layout(rect=(0, 0, 1, 0.98))
fig.savefig(OUTPUT_DIR / 'linear_vs_one_loop_panels.png', dpi=180, bbox_inches='tight')
plt.show()

## 7. Pulle

In [ ]:
fig, axes = plt.subplots(4, 2, figsize=(15, 14), sharex=True, sharey=True)
axes = axes.ravel()
for axis, z in zip(axes, dataset.z_unique):
    mask = np.isclose(dataset.z, z); order = np.argsort(dataset.k_velocity[mask]); kv = dataset.k_velocity[mask][order]
    for mode, style in [('linear', '--'), ('one_loop', '-')]:
        pull = (predictions[mode][mask] - dataset.p1d[mask]) / sigma[mask]
        axis.plot(kv, pull[order], style, marker='o', ms=3, label=mode)
    axis.axhline(0, color='black', lw=0.8); axis.axhline(2, color='0.5', ls=':'); axis.axhline(-2, color='0.5', ls=':')
    axis.set_xscale('log'); axis.set_title(f'z={z:.1f}'); axis.set_ylabel('pull')
    axis.legend(fontsize=8)
for axis in axes[len(dataset.z_unique):]: axis.set_visible(False)
for axis in axes[-2:]: axis.set_xlabel(r'$k_v$ [s/km]')
fig.tight_layout(); fig.savefig(OUTPUT_DIR / 'pulls.png', dpi=180, bbox_inches='tight'); plt.show()

## 8. Surowe kanały SPT

Linia pionowa oznacza `k_trust`. Nie jest to miejsce wyłączenia one-loop.

In [ ]:
iz = 0
fig, axes = plt.subplots(2, 3, figsize=(15, 8), sharex=True)
for channel, name in enumerate(CHANNELS):
    tree = theory.p_tree[iz]
    axes[0, channel].plot(theory.k_loop, theory.channels_one_loop[iz, :, channel] / tree)
    axes[0, channel].axhline(1, color='black', lw=0.8)
    axes[1, channel].plot(theory.k_loop, theory.p22[iz, :, channel] / tree, label='P22/tree')
    axes[1, channel].plot(theory.k_loop, theory.p13[iz, :, channel] / tree, label='P13/tree')
    axes[1, channel].plot(theory.k_loop, (theory.p22[iz, :, channel] + theory.p13[iz, :, channel]) / tree, lw=2, label='sum/tree')
    for axis in axes[:, channel]:
        axis.axvline(numerics['k_trust'], color='black', ls=':', lw=1); axis.set_xscale('log'); axis.set_title(name)
    axes[1, channel].set_xlabel(r'$k$ [h/Mpc]'); axes[1, channel].legend(fontsize=8)
axes[0, 0].set_ylabel('P_XY / tree'); axes[1, 0].set_ylabel('loop component / tree')
fig.suptitle(f'{MODEL_LABEL}, z={theory.z[iz]:.1f}: raw SPT'); fig.tight_layout(rect=(0, 0, 1, 0.96))
fig.savefig(OUTPUT_DIR / 'raw_spt_channels.png', dpi=180, bbox_inches='tight'); plt.show()

## 9. Zapis krzywych i lista wyników

In [ ]:
np.savez_compressed(OUTPUT_DIR / 'best_fit_curves.npz', z=dataset.z, k_velocity=dataset.k_velocity, data=dataset.p1d, sigma=sigma, linear=predictions['linear'], one_loop=predictions['one_loop'])
print('Zapisano:')
for path in sorted(OUTPUT_DIR.iterdir()):
    print(' -', path.name)